In [1]:
import sys
import os
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath('../src'))
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from Transformer import Transformer
from PreProcessorNN import PreProcessorNN
from PipelineBuilder import PipelineBuilder
from CrossValidation import CrossValidation

In [2]:
data_loader = DataLoader("../data/train.csv", "../data/test.csv")
train, test = data_loader.load()

In [3]:
train.shape, test.shape

((42000, 785), (28000, 784))

In [4]:
train.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
train.shape

(42000, 785)

In [6]:
train.describe()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,42000.000000,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,...,42000.000000,42000.000000,42000.000000,42000.00000,42000.000000,42000.000000,42000.0,42000.0,42000.0,42000.0
mean,4.456643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.219286,0.117095,0.059024,0.02019,0.017238,0.002857,0.0,0.0,0.0,0.0
std,2.887730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.312890,4.633819,3.274488,1.75987,1.894498,0.414264,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,254.000000,254.000000,253.000000,253.00000,254.000000,62.000000,0.0,0.0,0.0,0.0


In [7]:
train.isna().sum().sum() + test.isna().sum().sum()

np.int64(0)

No NANs in the train and test datasets

In [8]:
data_splitter = DataSplitter(target_column="label")
X_train, X_test, y_train, y_test = data_splitter.split(train)
X, y = train.drop(columns=['label']), train['label']
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0]/(X_train.shape[0] + X_test.shape[0])

((33600, 784), (8400, 784), (33600,), (8400,), 0.8)

In [9]:
pipeline = Pipeline([
    #('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
score = accuracy_score(y_test, y_pred)
score

0.9648809523809524

In [10]:
pipeline_builder = PipelineBuilder(train)
pipeline = pipeline_builder.build(model_name="KNN")
n_folds = 5
cv = CrossValidation(n_folds, pipeline)
cv_scores = cv.evaluate(X, y)
print(cv_scores)

{'mean_rmse': np.float64(0.9381666666666668), 'std_rmse': np.float64(0.002807707241620586)}


In [11]:
## Hyper-parameter tunning
param_grid = {
    "model__n_neighbors": [1, 2, 3, 4, 5, 8, 10, 13, 15, 20, 30],
}
search_knn = cv.hyper_param_tune(X, y, param_grid)
results = pd.DataFrame(search_knn.cv_results_)
results = results.sort_values("mean_test_score", ascending=False)
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
2,11.918753,1.567735,48.545673,1.463653,3,{'model__n_neighbors': 3},0.942143,0.938810,0.934643,0.939405,0.940000,0.939000,0.002452,1
4,12.050762,1.104401,49.262665,0.859938,5,{'model__n_neighbors': 5},0.939167,0.936548,0.933810,0.939167,0.942143,0.938167,0.002808,2
3,11.705527,0.970806,49.195645,1.080847,4,{'model__n_neighbors': 4},0.939048,0.935000,0.935714,0.938214,0.939643,0.937524,0.001840,3
0,12.457392,1.137366,50.471958,2.182703,1,{'model__n_neighbors': 1},0.938333,0.936786,0.934643,0.937738,0.938333,0.937167,0.001383,4
5,11.713836,1.137799,48.936967,1.406065,8,{'model__n_neighbors': 8},0.936071,0.933452,0.932619,0.936310,0.935714,0.934833,0.001503,5
6,12.968332,0.637782,50.579089,0.404152,10,{'model__n_neighbors': 10},0.934524,0.933214,0.930357,0.934643,0.936548,0.933857,0.002048,6
7,12.138459,0.584710,47.645103,0.848922,13,{'model__n_neighbors': 13},0.932619,0.929048,0.926310,0.932857,0.934048,0.930976,0.002869,7
8,12.836048,0.428353,48.920120,1.351153,15,{'model__n_neighbors': 15},0.931667,0.926786,0.925238,0.931071,0.933571,0.929667,0.003135,8
1,12.874730,1.361005,48.558886,2.346904,2,{'model__n_neighbors': 2},0.929524,0.925952,0.925952,0.926667,0.927381,0.927095,0.001325,9
9,12.973113,0.782358,50.510812,1.468559,20,{'model__n_neighbors': 20},0.925595,0.923214,0.920595,0.927976,0.929405,0.925357,0.003180,10


In [12]:
preds = search_knn.best_estimator_.predict(test)
submission = pd.DataFrame({"ImageId": test.index+1, "Label": preds})
submission.to_csv("../data/submission_knn.csv", index=False)

NameError: name 'search_rf' is not defined